# Data Ingestion

In [1]:
from langchain_core.documents import Document


In [2]:
import os 
os.makedirs("../data",exist_ok=True)

In [3]:
import fitz
from langchain_community.document_loaders import PyMuPDFLoader
def extract_tables_as_documents(pdf_path):
    doc = fitz.open(pdf_path)
    table_documents = []

    def table_to_markdown(table_data):
        md = ""
        header = table_data[0]
        md += "| " + " | ".join(str(x) if x else "" for x in header) + " |\n"
        md += "| " + " | ".join(["---"] * len(header)) + " |\n"

        for row in table_data[1:]:
            md += "| " + " | ".join(str(x) if x else "" for x in row) + " |\n"

        return md

    for page_number, page in enumerate(doc):
        tables = page.find_tables()

        if tables.tables:
            for table_index, table in enumerate(tables.tables):
                data = table.extract()

                if len(data) > 2:
                    markdown_table = table_to_markdown(data)

                    table_documents.append(
                        Document(
                            page_content=markdown_table,
                            metadata={
                                "source": pdf_path,
                                "page": page_number,
                                "type": "table",
                                "table_index": table_index
                            }
                        )
                    )

    return table_documents


/home/ssp/ML/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name,use_auth_token=False)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


/home/ssp/ML/venv/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|█| 103/103 [00:00<00:00, 559.68it/s, Materializing param=pooler.dense.weigh
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [6]:
class VectorStore:
    def __init__(self,collection_name:str="BNSS_PDF_Store",persist_directory:str="data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.collection =None
        self.client=None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF Embeddings for BNSS"}
            )
        except Exception as e:
            print(f"Error Creating Store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

loader = PyMuPDFLoader("data/250884_2_english_01042024.pdf")
pdf_docs = loader.load()

print("Raw pages:", len(pdf_docs))  

chunks = split_documents(pdf_docs)

table_docs = extract_tables_as_documents("data/250884_2_english_01042024.pdf")

all_docs = chunks + table_docs

texts = [doc.page_content for doc in all_docs]
embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(all_docs, embeddings)


Raw pages: 249
Split 249 documents into 1132 chunks

Example chunk:
Content: vlk/kkj.k
EXTRAORDINARY
Hkkx  II — [k.M 1
PART II — Section 1
izkf/kdkj ls izdkf'kr
PUBLISHED  BY  AUTHORITY
lañ   54]
ubZ fnYyh] lkseokj] fnlEcj 25] 2023@ikS"k 4] 1945 ¼'kd½
No. 54]
NEW DELHI, MONDAY...
Metadata: {'producer': 'iTextSharp™ 5.5.13.1 ©2000-2019 iText Group NV (AGPL-version)', 'creator': '', 'creationdate': '2023-12-25T21:46:27+05:30', 'source': 'data/250884_2_english_01042024.pdf', 'file_path': 'data/250884_2_english_01042024.pdf', 'total_pages': 249, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-12-25T21:48:28+05:30', 'trapped': '', 'modDate': "D:20231225214828+05'30'", 'creationDate': "D:20231225214627+05'30'", 'page': 0}
Consider using the pymupdf_layout package for a greatly improved page layout analysis.


Batches: 100%|███████████████████████████████████████████████████| 36/36 [00:45<00:00,  1.27s/it]


embeddings with shape: (1132, 384)
Adding 1132 documents to vector store...
Successfully added 1132 documents to vector store
Total documents in collection: 1132


In [9]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|███████████████████████████████████████████████████| 36/36 [00:55<00:00,  1.53s/it]


embeddings with shape: (1132, 384)
Adding 1132 documents to vector store...
Successfully added 1132 documents to vector store
Total documents in collection: 2264


In [10]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
            )

            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'distance': distance,
                        'rank': i + 1
                    })
            
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [11]:
results = rag_retriever.retrieve("What is Section 32 in BNSS?",)
results

Batches: 100%|█████████████████████████████████████████████████████| 1/1 [00:00<00:00, 92.44it/s]

embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c26d3a76_364',
  'content': 'section 78, section 79, section 143, section 199 or section 200 of the Bharatiya Nyaya\nSanhita, 2023.\n(2) No Court shall take cognizance of any offence alleged to have been committed by\nany member of the Armed Forces of the Union while acting or purporting to act in the\ndischarge of his official duty, except with the previous sanction of the Central Government.\n(3) The State Government may, by notification, direct that the provisions of\nsub-section (2) shall apply to such class or category of the members of the Forces charged\nwith the maintenance of public order as may be specified therein, wherever they may be\nserving, and thereupon the provisions of that sub-section will apply as if for the expression\n"Central Government" occurring therein, the expression "State Government" were\nsubstituted.\n(4) Notwithstanding anything contained in sub-section (3), no Court shall take\ncognizance of any offence, alleged to have been committed by a

In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0,api_key="")

class BNSS_QA_System:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template("""
        You are a legal assistant specializing in the Bharatiya Nagarik Suraksha Sanhita (BNSS), 2023.
        Answer the user's question based strictly on the provided context from the Act.
        If the information is not present in the context, explicitly state that the provided documents do not contain the answer.
        
        Context:
        {context}
        
        Question: {question}
        
        Answer:""")

    def format_context(self, docs):
        formatted_parts = []
        for d in docs:
            source = d['metadata'].get('source', 'Unknown')
            page = d['metadata'].get('page', 'Unknown')
            content = d['content']
            formatted_parts.append(f"Source: {source}, Page: {page}\nContent: {content}")
        return "\n\n---\n\n".join(formatted_parts)

    def ask(self, query):
        retrieved_docs = self.retriever.retrieve(query)
        context = self.format_context(retrieved_docs)
        chain = self.prompt | self.llm | StrOutputParser()
        response = chain.invoke({"context": context, "question": query})
        return response, retrieved_docs

qa_system = BNSS_QA_System(rag_retriever, llm)

In [16]:
query = "I have a problem with illegal hoardings on the road in my area what article should i quote and which authority to mail "
answer, docs = qa_system.ask(query)

print(f"QUERY: {query}\n")
print(f"ANSWER: {answer}\n")
print("SOURCES USED:")
for doc in docs:
    print(f"- Page {doc['metadata'].get('page', 'N/A')}: {doc['content'][:100]}...")

Batches: 100%|█████████████████████████████████████████████████████| 1/1 [00:00<00:00, 78.45it/s]


embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
QUERY: I have a problem with illegal hoardings on the road in my area what article should i quote and which authority to mail 

ANSWER: Based on the provided context:

For a problem with illegal hoardings on the road in your area, you could refer to **section 163** of the Bharatiya Nagarik Suraksha Sanhita, 2023, as mentioned in FORM No. 25, which deals with a "MAGISTRATE'S ORDER TO PREVENT OBSTRUCTION, RIOT, ETC." The form specifically mentions situations like throwing earth and stones upon an adjoining public road "so as to occasion risk of obstruction to persons using the road."

The authority mentioned in relation to this order is a **Magistrate**.

SOURCES USED:
- Page 27: (c) to take possession of any property or article therein found which he
reasonably suspects to be s...
- Page 27: (c) to take possession of any property or article therein found which he
reasonably suspects to be s...
- Page 213: FORM No. 2